# RAMP Time Series Generation — Norte Amazonia Bolivia

This notebook builds the hourly time-series input files (`Time_series_C*.csv`) required by EnergyScope for each cluster.

For each cluster, RAMP minute-level load data is averaged to 8760 hourly values, summed across municipalities, and normalized to a fractional profile (sum = 1). Renewable capacity factors (PV, wind, solar) come from Renewables.ninja outputs. Mobility profiles are shared across all clusters.

## 1. Setup — paths, clusters, and column groups

`ELECTRICITY_COLS` lists every RAMP column that draws electrical power — combined, they shape the cluster's total electrical load profile.  
`SPACE_COOLING_COLS` is a subset covering only thermal comfort and air conditioning, used to build a separate space-cooling profile.

In [5]:
import os
import pandas as pd
import numpy as np

RAMP_DIR       = "data ramp"
RENEWABLES_DIR = "../renewable ninja/solar and wind/output"
OUTPUT_DIR     = "output_energyscope"

CLUSTERS = {
    1: ["Exaltación", "Reyes", "Santa_Rosa_Beni", "Ixiamas"],
    2: ["Bolpebra"],
    3: ["Guayaramerín", "Riberalta", "Puerto_Gonzalo_Moreno"],
    4: ["Bella_Flor", "Filadelfia", "Ingavi", "Nueva_Esperanza", "Porvenir", "Puerto_Rico", "San_Lorenzo", "San_Pedro", "Santa_Rosa_Pando", "Santos_Mercado", "Sena", "Villa_Nueva"],
    5: ["Cobija"],
}

# All electrical end-uses — combined, they form the cluster's electricity demand shape
ELECTRICITY_COLS = [
    "sufficiency_illumination", "sufficiency_ICT", "sufficiency_cold_storage",
    "sufficiency_thermal_comfort", "public_lighting_illumination",
    "big_school_illumination", "big_school_ICT", "big_school_cold_storage",
    "big_school_space_cooling",
    "health_center_illumination", "health_center_ICT", "health_center_cold_storage",
    "health_center_space_cooling", "health_center_water_supply", "health_center_medical_equip",
    "entertainment_business_illumination", "entertainment_business_ICT",
    "entertainment_business_cold_storage",
    "restaurant_illumination", "restaurant_cold_storage", "restaurant_kitchen",
    "store_illumination", "store_ICT", "store_cold_storage",
    "workshop_illumination", "workshop_ICT", "workshop_machinery",
    "rice_processing_rice_processing",
]

# Subset for the space-cooling profile (thermal comfort + school/health AC only)
SPACE_COOLING_COLS = [
    "sufficiency_thermal_comfort",
    "big_school_space_cooling",
    "health_center_space_cooling",
]

HOURS = pd.date_range("2019-01-01", periods=8760, freq="h", tz="UTC")

## 2. Helper functions

- **`load_ramp_hourly(muni, cols)`** — reads a municipality's RAMP CSV (1-minute steps, 525,600 rows), keeps only the requested columns, and averages every 60 consecutive rows into one hourly value, giving an 8760-row result
- **`cluster_profile(munis, cols)`** — sums the raw watt values across all municipalities in a cluster, then divides by the grand total so the result sums to exactly 1 (a fractional time distribution required by EnergyScope)

> Averaging 60 minutes → 1 hour preserves the load shape while reducing data size 60×. No unit conversion is needed because the profile is normalized afterwards.

In [6]:
def load_ramp_hourly(muni, cols):
    """Read a municipality's RAMP CSV and return an 8760-row DataFrame of hourly averages."""
    path = f"{RAMP_DIR}/{muni}/load_curve_energy_service_full_year_Norte_Amazonia.csv"
    if not os.path.exists(path):
        print(f"  Warning: file not found for {muni}")
        return None

    # Read only the header first to avoid loading unused columns into memory
    available = pd.read_csv(path, nrows=0).columns.tolist()
    valid_cols = [c for c in cols if c in available]
    if not valid_cols:
        return None

    df = pd.read_csv(path, usecols=valid_cols).fillna(0)
    # Reshape 525,600 minute rows into 8760 hourly rows by averaging each 60-row block
    hourly = pd.DataFrame({c: df[c].values.reshape(8760, 60).mean(axis=1) for c in valid_cols})
    del df  # free the large minute-level array immediately
    return hourly


def cluster_profile(munis, cols):
    """Sum raw watts across all municipalities in a cluster, then normalize to a unit profile (sum = 1)."""
    total = np.zeros(8760)
    for muni in munis:
        hourly = load_ramp_hourly(muni, cols)
        if hourly is not None:
            total += hourly.sum(axis=1).values
            del hourly  # free memory after accumulating each municipality

    if total.sum() == 0:
        print("  Warning: zero profile — using flat 1/8760")
        return np.full(8760, 1 / 8760)

    return total / total.sum()

## 3. Build the time series for each cluster

For each cluster:
1. Build the electricity and space-cooling load profiles from RAMP data
2. Read renewable capacity factors (PV, wind, solar) from the Renewables.ninja outputs
3. Combine everything into a single 8760-row DataFrame and save it

Mobility profiles are read once from a shared CSV and are identical for every cluster.

> `WIND_OFFSHORE` copies `WIND_ONSHORE` values — no offshore resource exists in this landlocked region, but the column is required by EnergyScope.  
> `HYDRO_DAM`, `HYDRO_RIVER`, `TIDAL`, and `CSP` are set to 0.0001 to satisfy EnergyScope's format without enabling those technologies.

In [7]:
# Mobility profiles are the same for all clusters — load and normalize once
mob = pd.read_csv("data/Time_series.csv", sep=";", index_col=0)
mob_passenger = mob["MOBILITY_PASSENGER"].values / mob["MOBILITY_PASSENGER"].sum()
mob_freight   = mob["MOBILITY_FREIGHT"].values   / mob["MOBILITY_FREIGHT"].sum()

print("--- BUILDING TIME SERIES ---")

for k, munis in CLUSTERS.items():
    elec = cluster_profile(munis, ELECTRICITY_COLS)
    sc   = cluster_profile(munis, SPACE_COOLING_COLS)
    ren  = pd.read_csv(f"{RENEWABLES_DIR}/renewables_C{k}.csv")

    ts = pd.DataFrame({
        "ELECTRICITY":        elec,
        "SPACE_COOLING":      sc,
        "MOBILITY_PASSENGER": mob_passenger,
        "MOBILITY_FREIGHT":   mob_freight,
        "PV":                 ren["PV"].values,
        "WIND_ONSHORE":       ren["WIND_ONSHORE"].values,
        "WIND_OFFSHORE":      ren["WIND_ONSHORE"].values,  # no offshore resource in this region
        "HYDRO_DAM":          0.0001,
        "HYDRO_RIVER":        0.0001,
        "TIDAL":              0.0001,
        "SOLAR":              ren["SOLAR"].values,
        "CSP":                0.0001,
    }, index=HOURS)
    ts.index.name = ""

    out_path = f"{OUTPUT_DIR}/C{k}/Time_series.csv"
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    ts.to_csv(out_path, sep=";", date_format="%Y-%m-%d %H:%M:%S+00:00")
    print(f"Cluster {k} ({len(munis)} municipalities) — saved to {out_path}")

--- BUILDING TIME SERIES ---
Cluster 1 (4 municipalities) — saved to output_energyscope/C1/Time_series.csv
Cluster 2 (1 municipalities) — saved to output_energyscope/C2/Time_series.csv
Cluster 3 (3 municipalities) — saved to output_energyscope/C3/Time_series.csv
Cluster 4 (12 municipalities) — saved to output_energyscope/C4/Time_series.csv
Cluster 5 (1 municipalities) — saved to output_energyscope/C5/Time_series.csv


## 4. Verification

Check that each normalized profile sums to 1.0, confirming correct normalization.  
Renewable columns (PV, WIND, SOLAR) will sum to their total annual capacity-factor hours — not 1.

In [8]:
print("--- VERIFICATION (normalized profiles should sum to 1.0) ---")
for k in CLUSTERS:
    ts = pd.read_csv(f"{OUTPUT_DIR}/C{k}/Time_series.csv", sep=";", index_col=0)
    sums = ts.sum().round(4)
    print(f"C{k}: {sums.to_dict()}")

--- VERIFICATION (normalized profiles should sum to 1.0) ---
C1: {'ELECTRICITY': 1.0, 'SPACE_COOLING': 1.0, 'MOBILITY_PASSENGER': 1.0, 'MOBILITY_FREIGHT': 1.0, 'PV': 1523.6071, 'WIND_ONSHORE': 511.7734, 'WIND_OFFSHORE': 511.7734, 'HYDRO_DAM': 0.876, 'HYDRO_RIVER': 0.876, 'TIDAL': 0.876, 'SOLAR': 1891.2077, 'CSP': 0.876}
C2: {'ELECTRICITY': 1.0, 'SPACE_COOLING': 1.0, 'MOBILITY_PASSENGER': 1.0, 'MOBILITY_FREIGHT': 1.0, 'PV': 1498.1877, 'WIND_ONSHORE': 477.9004, 'WIND_OFFSHORE': 477.9004, 'HYDRO_DAM': 0.876, 'HYDRO_RIVER': 0.876, 'TIDAL': 0.876, 'SOLAR': 1781.2272, 'CSP': 0.876}
C3: {'ELECTRICITY': 1.0, 'SPACE_COOLING': 1.0, 'MOBILITY_PASSENGER': 1.0, 'MOBILITY_FREIGHT': 1.0, 'PV': 1531.167, 'WIND_ONSHORE': 528.969, 'WIND_OFFSHORE': 528.969, 'HYDRO_DAM': 0.876, 'HYDRO_RIVER': 0.876, 'TIDAL': 0.876, 'SOLAR': 1930.4404, 'CSP': 0.876}
C4: {'ELECTRICITY': 1.0, 'SPACE_COOLING': 1.0, 'MOBILITY_PASSENGER': 1.0, 'MOBILITY_FREIGHT': 1.0, 'PV': 1440.997, 'WIND_ONSHORE': 426.0345, 'WIND_OFFSHORE': 4